# 01 - ATT&CK Data Exploration

Skeleton notebook for exploring the loaded MITRE ATT&CK Enterprise snapshot and the derived tables (`techniques`, `apt_groups`, `apt_technique`, `technique_coverage`, ...) held in `db/coverage.db` (or whatever `DATABASE_URL` points at).

**Contents**

1. Environment bootstrap - makes `src/` importable, defaults `DATABASE_URL` to the repo-local SQLite file
2. Provenance header - library/bundle versions from `attack_meta` vs locally installed packages
3. Per-table row counts
4. Tactic distribution (pandas + Plotly bar)
5. Top-15 groups by distinct technique usage

Cells ship un-executed. Run All once the ATT&CK loader has populated the database (see `README.md`). Expected magnitudes live in `research/p3-wave1-full-results.md`.

In [ ]:
# --- Bootstrap ----------------------------------------------------------------
# Make `src/` importable regardless of kernel CWD and default DATABASE_URL to the
# repo-local SQLite file. Idempotent: safe to re-run.
import os
import sys
from pathlib import Path

_cwd = Path.cwd()
REPO_ROOT = _cwd if (_cwd / 'src').is_dir() else _cwd.parent
if not (REPO_ROOT / 'src').is_dir():
    raise RuntimeError(
        f'Could not locate repo root containing src/ (started from {_cwd}); '
        'launch Jupyter from the repository root or from notebooks/'
    )

_SRC = str(REPO_ROOT / 'src')
if _SRC not in sys.path:
    sys.path.insert(0, _SRC)

# session.database_url() honours an explicit DATABASE_URL / .env; only fill the
# default so launching from notebooks/ still finds db/coverage.db.
os.environ.setdefault(
    'DATABASE_URL',
    f"sqlite:///{(REPO_ROOT / 'db' / 'coverage.db').as_posix()}",
)

_db_url = os.environ['DATABASE_URL']
print(f'repo root    : {REPO_ROOT}')
print(f'DATABASE_URL : {_db_url}')

## Provenance

The single-row `attack_meta` table pins which `mitreattack-python` release and Enterprise bundle produced this snapshot. Compare it against the locally installed versions before trusting any counts.

In [ ]:
# --- Provenance ---------------------------------------------------------------
import importlib.metadata as importlib_metadata

import pandas as pd
import plotly.express as px
from sqlalchemy import inspect, text

from database.session import make_engine

engine = make_engine()
_inspector = inspect(engine)

meta_row = None
if _inspector.has_table('attack_meta'):
    with engine.connect() as conn:
        meta_row = conn.execute(
            text(
                'SELECT library_version, bundle_version, bundle_date, '
                'object_count, loaded_at FROM attack_meta '
                'ORDER BY id DESC LIMIT 1'
            )
        ).mappings().first()

if meta_row is None:
    print('WARNING: attack_meta missing or empty - run the ATT&CK loader first.')
else:
    print('== Snapshot (attack_meta) ==')
    print('  mitreattack-python :', meta_row['library_version'])
    print('  enterprise bundle  :', meta_row['bundle_version'], '-', meta_row['bundle_date'])
    print('  objects loaded     :', meta_row['object_count'])
    print('  loaded_at          :', meta_row['loaded_at'])

print('\n== Local packages ==')
for _dist in ('mitreattack-python', 'pandas', 'plotly', 'scikit-learn', 'sqlalchemy'):
    try:
        print(f'  {_dist:<18}', importlib_metadata.version(_dist))
    except importlib_metadata.PackageNotFoundError:
        print(f'  {_dist:<18}', 'NOT INSTALLED')

In [ ]:
# --- Table counts -------------------------------------------------------------
# Row counts for every core table. On a fresh clone (tables not created yet)
# missing tables are reported as '-' instead of raising.
from sqlalchemy import func, select

from database.models import (
    Analytic,
    AptGroup,
    AptTechnique,
    AttackMeta,
    DataSourceType,
    DetectionStrategy,
    SimulationResult,
    SimulationTechnique,
    Technique,
    TechniqueCoverage,
)

_TABLES = [
    ('techniques', Technique),
    ('detection_strategies', DetectionStrategy),
    ('analytics', Analytic),
    ('apt_groups', AptGroup),
    ('apt_technique', AptTechnique),
    ('datasource_types', DataSourceType),
    ('technique_coverage', TechniqueCoverage),
    ('simulation_results', SimulationResult),
    ('simulation_techniques', SimulationTechnique),
    ('attack_meta', AttackMeta),
]

counts = {}
for _label, _model in _TABLES:
    if not _inspector.has_table(_model.__tablename__):
        counts[_label] = None
    else:
        with engine.connect() as conn:
            counts[_label] = conn.execute(
                select(func.count()).select_from(_model)
            ).scalar_one()

for _label, _n in counts.items():
    print(f'{_label:>22}: ', '-' if _n is None else _n)

# Load the technique table once for reuse below. When it is missing or empty,
# degrade to empty frames so the rest of the notebook still renders.
_TECH_COLS = [
    'stix_id', 'attack_id', 'name', 'tactics', 'platforms',
    'is_subtechnique', 'revoked',
]
if counts.get('techniques'):
    tech_df = pd.read_sql(
        select(
            Technique.stix_id,
            Technique.attack_id,
            Technique.name,
            Technique.tactics,
            Technique.platforms,
            Technique.is_subtechnique,
            Technique.revoked,
        ),
        engine,
    )
    alive = tech_df[~tech_df['revoked']].copy()
    print()
    print('alive techniques       :', len(alive))
    print('revoked techniques     :', int(tech_df['revoked'].sum()))
    print('sub-techniques         :', int(alive['is_subtechnique'].sum()))
    print('zero-tactic techniques :',
          int((alive['tactics'].map(lambda ts: len(ts or []) == 0)).sum()))
    print('distinct platforms     :',
          len({p for ps in alive['platforms'] for p in (ps or [])}))
else:
    tech_df = pd.DataFrame(columns=_TECH_COLS)
    alive = tech_df
    print('\nEmpty or missing techniques table - cells below render placeholders.')

## Tactic distribution

Alive (non-revoked) techniques per Enterprise tactic, in the canonical ATT&CK order used by the coverage engine. A multi-tactic technique counts once per tactic - same denominator semantics as `coverage_by_tactic`. Zero-tactic rows (e.g. while v19-only kill-chain phases such as `stealth` / `defense-impairment` await the loader alias fix) are excluded from the bars.

In [ ]:
# --- Tactic distribution -------------------------------------------------------
CANONICAL_TACTICS = [
    'reconnaissance', 'resource-development', 'initial-access', 'execution',
    'persistence', 'privilege-escalation', 'defense-evasion',
    'credential-access', 'discovery', 'lateral-movement', 'collection',
    'command-and-control', 'exfiltration', 'impact',
]

_exploded = alive.explode('tactics')
_exploded = _exploded[_exploded['tactics'].isin(CANONICAL_TACTICS)]
tactic_counts = (
    _exploded['tactics']
    .value_counts()
    .reindex(CANONICAL_TACTICS, fill_value=0)
    .astype(int)
    .rename('alive_techniques')
)
display(tactic_counts.to_frame())

fig_tactics = px.bar(
    x=tactic_counts.index,
    y=tactic_counts.to_numpy(),
    labels={'x': 'tactic', 'y': 'alive techniques'},
    title='ATT&CK Enterprise techniques per tactic (non-revoked)',
)
fig_tactics.update_layout(
    xaxis={'categoryorder': 'array', 'categoryarray': CANONICAL_TACTICS}
)
fig_tactics.show()

## Group -> technique usage

Top 15 threat groups by number of distinct associated techniques (`apt_technique` join). Sanity-checks that relationships loaded, and feeds the simulation-driven coverage views later. Ties break deterministically by `group_id`.

In [ ]:
# --- Groups by technique usage --------------------------------------------------
from sqlalchemy import desc

if not _inspector.has_table('apt_groups'):
    group_usage = pd.DataFrame(columns=['group_id', 'group_name', 'techniques'])
else:
    group_usage = pd.read_sql(
        select(
            AptGroup.group_id,
            AptGroup.name.label('group_name'),
            func.count(AptTechnique.technique_stix_id.distinct()).label('techniques'),
        )
        .join(AptTechnique, AptTechnique.group_stix_id == AptGroup.stix_id)
        .group_by(AptGroup.group_id, AptGroup.name)
        .order_by(desc('techniques'), AptGroup.group_id)
        .limit(15),
        engine,
    )

if group_usage.empty:
    print('No group data loaded yet (apt_groups / apt_technique empty or missing).')
else:
    display(group_usage)
    labelled = group_usage.assign(
        label=group_usage['group_id'] + ' - ' + group_usage['group_name']
    )
    fig_groups = px.bar(
        labelled,
        y='label',
        x='techniques',
        orientation='h',
        title='Top 15 groups by distinct techniques used',
    )
    fig_groups.update_yaxes(autorange='reversed')
    fig_groups.show()

## Notes and next steps

- Shapes above should be compared against the CORE-ROW AGGREGATE snapshot in `research/p3-wave1-full-results.md`; note that file reports Windows-*scoped* denominators, whereas this notebook shows the whole Enterprise set.
- Coverage-specific exploration (per-datasource detectability, gaps, clusters) belongs in follow-up notebooks once the Phase-3 coverage engine populates `technique_coverage`.
- The database is regenerated wholesale by the loader; do not edit rows by hand.